# Trace and validate the pipeline

Linden Facilities needs 24 hourly estimates. Follow the five stage contracts before making a bounded validation change.

**Local:** [set up once](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/docs/setup.md), select the module `.venv` kernel, then run cells in order. **Colab:** CPU, runtime version **2026.07** (Python 3.12). If setup asks for a restart after installing dependencies, use **Restart session**, then run again. No API key is needed. Setup prints a new temporary output folder; save your work before ending the session.

Use this one notebook throughout the sprint: predict → run the guided cells → add your answers in **Your turn** → finish the checkpoint. Campus notes explain the same work; you do not repeat it in a second notebook.

<details><summary>Campus notes for this sprint</summary>

- [From notebook to pipeline](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-1/01-from-notebook-to-pipeline/lesson.md)
- [Reading the provided pipeline scaffold](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-1/02-reading-the-provided-pipeline-scaffold/lesson.md)
- [Adding and validating a bounded pipeline stage](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-1/03-adding-and-validating-a-bounded-pipeline-stage/lesson.md)
- [Explain-the-code checkpoint](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-1/04-explain-the-code-checkpoint/lesson.md)
- [Batch vs real-time inference tradeoffs](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-1/05-batch-vs-real-time-inference-tradeoffs/lesson.md)
- [Pipeline extension deliverable](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-1/06-pipeline-extension-deliverable/lesson.md)

</details>

In [1]:
from pathlib import Path
import json, tempfile, sys, subprocess
PIN = {'package': 'campus-mlops', 'version': '0.1.1', 'code_revision': 'sha256:a7a3061850f1f1eaa158754e719262b26a000af73019ada402be743c6f66e6c2', 'repository': 'https://github.com/richhiey/ml-llm-ops.git', 'source_commit': 'f1ae85914f49a4f1c4bdd6a0ca62c4c4076158ea', 'package_path': 'module-a'}
assert sys.version_info[:2] == (3, 12), 'Use Python 3.12; see the linked setup guide.'
if 'google.colab' in sys.modules:
    # Start before importing numerical packages: installed files cannot replace live modules.
    expected = {'numpy':'2.3.3', 'pandas':'2.3.3', 'sklearn':'1.7.2', 'mlflow':'3.5.1'}
    stale = [name for name, version in expected.items()
             if name in sys.modules and getattr(sys.modules[name], '__version__', None) != version]
    checkout = Path('/content') / ('ml-llm-ops-' + PIN['source_commit'][:12])
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--no-checkout', PIN['repository'], str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', PIN['source_commit']], check=True)
    actual = subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'], text=True).strip()
    assert actual == PIN['source_commit']
    package = checkout / PIN['package_path']
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(package/'requirements-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest==8.4.2'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(package)], check=True)
    if stale:
        raise RuntimeError('Pinned dependencies are now installed. Choose Runtime > Restart session, '
                           'then Run all again. Keep this runtime: do NOT disconnect/delete it. '
                           'Loaded packages needing refresh: ' + ', '.join(stale))
    # A running kernel does not reload newly installed editable .pth files.
    sys.path.insert(0, str(package / 'src'))
import campus_mlops
from campus_mlops import *
from campus_mlops.lifecycle import code_revision, MODEL_NAME
PACKAGE=Path(campus_mlops.__file__).resolve().parents[2]
sys.path.insert(0,str(PACKAGE))
from learning.workspace import make_project, project_python, project_tests, course_cli, prepare_case
assert campus_mlops.__version__ == PIN['version'] == '0.1.1'
assert code_revision() == PIN['code_revision'], 'Helper changed: restore the pinned reference source.'
state = Path(tempfile.mkdtemp(prefix='campus-module-a-'))
print('Helper', campus_mlops.__version__, 'source commit:', PIN['source_commit'], 'state:', state)


Helper 0.1.1 source commit: f1ae85914f49a4f1c4bdd6a0ca62c4c4076158ea state: /var/folders/tc/l_pmm63j3jn1y1vyr31_4qhc0000gn/T/campus-module-a-pen7oq7y


## Inspect the stage boundaries
`ingest` reads the fixed snapshot. Features exclude target and partition. Expect 168 train, 48 validation and 24 final rows.

Map the returned DataFrame to ingestion, feature extraction, fitting and evaluation. `partition` selects rows; it never becomes a feature. Predict whether selecting features across all rows fits any statistics, then inspect `preprocess` in the linked source.

In [2]:
frame=ingest()
print(frame.groupby('partition').size())
display(frame.head())

partition
final          24
train         168
validation     48
dtype: int64


,row_id,temperature_c,hour,occupancy,demand_kwh,partition
0,D000,21.714,0,89,76.066729,validation
1,D001,-1.100,1,0,45.455758,validation
2,D002,9.338,2,12,44.943443,final
3,D003,16.650,3,67,69.456248,train
4,D004,27.623,4,34,45.034036,train


## Trace preparation, fitting and evaluation
The scaler belongs to the fitted model, so it sees training rows only. RMSE measures larger errors more strongly; MAE is diagnostic. Expect baseline validation RMSE about 1.443003925 kWh.

`features.loc[train_rows]` selects fitting inputs, and the matching target uses the same mask. The scaler learns only from those fitting rows. `evaluate` calls `predict` on held-out features; it does not fit again. Read the masks before treating the metric as a validation result.

In [3]:
from campus_mlops.pipeline import TARGET
config=Config()
features=preprocess(frame,config)
train_rows=frame.partition=='train'
valid_rows=frame.partition=='validation'
model=train(features.loc[train_rows],frame.loc[train_rows,TARGET],config)
metrics=evaluate(model,features.loc[valid_rows],frame.loc[valid_rows,TARGET])
print(features.columns.tolist(), metrics)
assert abs(metrics['rmse_kwh']-1.4430039250255964)<1e-6

['temperature_c', 'hour', 'occupancy'] {'rmse_kwh': 1.4430039250255964, 'mae_kwh': 1.2234272153578762}


## Inspect fitted state and row roles

Predict the three printed observations before running: fitting/validation counts, named fitted steps, and number of input features. Distinguish a configured alpha from statistics learned by fitting.

In [4]:
print(int(train_rows.sum()), int(valid_rows.sum()))
print(list(model.named_steps))
print(model.named_steps['scale'].n_features_in_)
assert not (train_rows & valid_rows).any()
assert features.index.equals(frame.index)

168 48
['scale', 'ridge']
3


## Read the extension, then test both branches
This is the worked reference. Explain every line before using it. It rejects invalid values and copies valid inputs. Enabling it must preserve valid outputs.

In [5]:
import inspect
print(inspect.getsource(validate_ranges))
import pandas as pd
pd.testing.assert_frame_equal(validate_ranges(frame),frame)
broken=frame.copy(); broken.loc[0,'hour']=24
try:
    validate_ranges(broken)
except ValueError as error:
    print('Expected rejection:', error)
else:
    raise AssertionError('Invalid hour was accepted')
assert run_pipeline(Config(range_guard=True))[1]==metrics

def validate_ranges(frame):
    """Bounded extension: reject invalid readings; never clip/drop/reorder valid rows."""
    validate_features(frame)
    for column, low, high in [("temperature_c", -30, 50), ("hour", 0, 23), ("occupancy", 0, 100)]:
        if not frame[column].between(low, high).all():
            raise ValueError(f"{column} outside [{low}, {high}]")
    if not (frame.hour % 1 == 0).all():
        raise ValueError("hour must be an integer")
    return frame.copy(deep=True)

Expected rejection: hour outside [0, 23]


## Complete the inference stage
The prepared bootstrap uses the same pipeline, logs it, registers and activates an approved model. Expect 24 finite output rows tied to the resolved registry version. Timing is observed, not a fixed benchmark.

Registration/selection here are prepared infrastructure so you can trace the fifth stage. Later lifecycle work explains those decisions. The input has no known target, so this job cannot establish prediction error for tomorrow.

In [6]:
ws=Workspace(state/'tracking')
record=ws.run(Config(range_guard=True),name='guarded-baseline')
version=register(ws,record['run_id'])
approve(ws,version,'Valid/rejected input checks and RMSE <= 5')
activate(ws,version,'Baseline for batch demonstration')
predictions,evidence=batch_predict(ws)
display(predictions.head()); print(evidence)
assert len(predictions)==24 and evidence['resolved_version']==version
assert evidence['complete_within_60s']

2026/10/19 17:38:02 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/19 17:38:02 INFO mlflow.store.db.utils: Updating database tables


2026-10-19 17:38:02 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 17:38:02 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


2026-10-19 17:38:02 INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step


...

2026-10-19 17:38:02 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 17:38:02 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


Successfully registered model 'dev.linden-demand'.
Created version '1' of model 'dev.linden-demand'.


,row_id,prediction_kwh
0,B00,47.639043
1,B01,48.413678
2,B02,49.188314
3,B03,49.962949
4,B04,50.737584


{'resolved_version': '1', 'run_id': '06adebae239e4ed8b668b6886414c959', 'model': 'dev.linden-demand', 'rows': 24, 'elapsed_seconds': 0.010045500006526709, 'rows_per_second': 2389.129459400413, 'complete_within_60s': True, 'input_sha256': '5b74deb237ea9a085666ab0b3f865db8c7c996b935b38b24e829567d89142ee6'}


## Check the delivered rows

Compare identifiers and finite outputs, then connect the returned source run to the logged execution. A matching row count alone could hide incorrect alignment.

In [7]:
from campus_mlops.pipeline import DATA
import numpy as np
planned=pd.read_csv(DATA/'batch-v1.csv')
assert planned.row_id.tolist()==predictions.row_id.tolist()
assert np.isfinite(predictions.prediction_kwh).all()
assert evidence['run_id']==record['run_id']
print('Aligned rows:',len(predictions),'complete within deadline:',evidence['complete_within_60s'])

Aligned rows: 24 complete within deadline: True


## Edit the validation stage

Make an actual source change in a new project. First predict what the finite-only starter does with `hour=24`. The focused test below intentionally fails (`DID NOT RAISE`); ordinary valid-data RMSE cannot reveal this missing rule. The original reference remains available.

In [8]:
project=make_project('stage-project')
project_tests(project,'tests/test_spine.py::test_contract_extension_and_unchanged_pipeline','-q',expected_returncode=1)
print('Edit target:',project/'src/campus_mlops/pipeline.py')

{
  "project_root": "/private/var/folders/tc/l_pmm63j3jn1y1vyr31_4qhc0000gn/T/linden-project-e58sx3w0/stage-project",
  "starting_stage": "finite-only; range checks deliberately absent",
  "reference_patch": "/private/var/folders/tc/l_pmm63j3jn1y1vyr31_4qhc0000gn/T/linden-project-e58sx3w0/stage-project/expert-stage.patch"
}



F                                                                        [100%]
=================================== FAILURES ===================================
________________ test_contract_extension_and_unchanged_pipeline ________________

    def test_contract_extension_and_unchanged_pipeline():
        raw = ingest()
        original = raw.copy(deep=True)
        checked = validate_ranges(raw)
        pd.testing.assert_frame_equal(raw, checked)
        pd.testing.assert_frame_equal(raw, original)
        assert raw.groupby('partition').size().to_dict() == {'final': 24, 'train': 168, 'validation': 48}
        base = run_pipeline(Config())[1]
        extended = run_pipeline(Config(range_guard=True))[1]
        assert base == extended
        assert extended['rmse_kwh'] < 5
        for col, bad in [('hour',24), ('hour',-1), ('hour',1.5), ('temperature_c',51), ('temperature_c',-31), ('occupancy',101), ('occupancy',-1), ('occupancy',np.nan), ('hour',np.inf)]:
            broken = raw.c

## Apply the explained expert change

Read the reference function printed earlier. It checks finite values, inclusive ranges and integer hours, then returns an unchanged copy. `.all()` requires every row to pass; placing `return` inside the loop would skip later fields. This cell replaces only that function in your project and prints the real diff. Explain each changed statement before running.

In [9]:
import inspect,difflib
target=project/'src/campus_mlops/pipeline.py'
before=target.read_text()
start=before.index('def validate_ranges(')
end=before.index('\n\ndef preprocess(',start)
after=before[:start]+inspect.getsource(validate_ranges).rstrip()+before[end:]
target.write_text(after)
print(''.join(difflib.unified_diff(before.splitlines(True),after.splitlines(True),fromfile='starter',tofile='edited')))

--- starter
+++ edited
@@ -53,9 +53,14 @@
 
 
 def validate_ranges(frame):
-    """Starting stage: finite features only; domain ranges are not implemented yet."""
-    return validate_features(frame).copy(deep=True)
-
+    """Bounded extension: reject invalid readings; never clip/drop/reorder valid rows."""
+    validate_features(frame)
+    for column, low, high in [("temperature_c", -30, 50), ("hour", 0, 23), ("occupancy", 0, 100)]:
+        if not frame[column].between(low, high).all():
+            raise ValueError(f"{column} outside [{low}, {high}]")
+    if not (frame.hour % 1 == 0).all():
+        raise ValueError("hour must be an integer")
+    return frame.copy(deep=True)
 
 def preprocess(frame, config):
     """Table -> unchanged feature order; scaler is fitted on training rows by train()."""



## Check the edited project

Execute the project in fresh processes so the notebook cannot reuse an earlier imported reference function. Expect the full suite to pass, unchanged valid-data metrics and a complete batch. Keep the printed project path: continue with this same source in later sprints.

In [10]:
project_tests(project,'-q')
project_python(project, "from campus_mlops import Config,run_pipeline; assert run_pipeline(Config())[1]==run_pipeline(Config(range_guard=True))[1]; print('Valid-data metrics unchanged')")
import contextlib,io
with contextlib.redirect_stdout(io.StringIO()):
    result=course_cli('bootstrap','--state',str(state/'edited-batch'),project=project)
completed=json.loads(result.stdout)
print('Edited-project batch:',completed['inference'])
print('Saved project:',project)

......                                                                   [100%]
=============================== warnings summary ===============================
tests/test_spine.py::test_reproduction_checks_transitive_environment_before_running
tests/test_spine.py::test_approval_rejects_final_partition_and_nonfinite_metric
tests/test_spine.py::test_approval_rejects_final_partition_and_nonfinite_metric
  /Users/richhiey/Desktop/projects/masterschool/lessons/mlops-llmops/shared/ml-llm-ops/module-a/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare inte

Valid-data metrics unchanged



Edited-project batch: {'resolved_version': '1', 'run_id': '9248c3a0361b44a88876c283b6edbff9', 'model': 'dev.linden-demand', 'rows': 24, 'elapsed_seconds': 0.007200833992101252, 'rows_per_second': 3332.94727059756, 'complete_within_60s': True, 'input_sha256': '5b74deb237ea9a085666ab0b3f865db8c7c996b935b38b24e829567d89142ee6'}
Saved project: /var/folders/tc/l_pmm63j3jn1y1vyr31_4qhc0000gn/T/linden-project-e58sx3w0/stage-project


## Your turn: explain and challenge the stage

Load `PACKAGE/exercises/generated-stage-candidate.py` as exercise material. Explain its statements and predict its behavior on a negative hour, fractional hour and a valid table. Run probes, check mutation, and correct the proposal in your project. Use the Campus practice hints only if stuck. In this notebook, justify batch inference for 24 planned hourly rows and the 60-second completion requirement; a request service is a comparison, not another build. Keep one short decision table with your code and observations.

In [11]:
import runpy
proposed_guard=runpy.run_path(str(PACKAGE/'exercises/generated-stage-candidate.py'))['proposed_guard']
supplier=pd.read_csv(PACKAGE/'exercises/harbor-intake.csv')
display(supplier)
print('Candidate loaded for your probes; source data has not been changed.')

,row_id,temperature_c,hour,occupancy
0,H-201,12,23.0,100.0
1,H-202,12,22.5,40.0
2,H-203,-30,0.0,0.0
3,H-204,51,12.0,20.0
4,H-205,20,12.0,-0.5


Candidate loaded for your probes; source data has not been changed.


## Sprint checkpoint

Use `PACKAGE/exercises/harbor-intake.csv` with your edited guard. Classify the supplier rows, preserve the raw file, test valid/invalid cases and confirm unchanged downstream behavior. Explain the stage and justify the chosen serving path. Complete the Campus client brief here; link the actual batch and test results. Do not create another notebook or a separate evidence report.

## Finish and keep your work

Save this notebook and the linked project/output files. Keep one sprint submission, reusing the work already completed. Temporary tracking directories disappear when the environment is discarded; save the files before closing it. Continue with the same project in the next sprint.